In [1]:
import os
import re
import pdfplumber
import pandas as pd
from pathlib import Path
from datetime import datetime

In [2]:
CITIES = [
    'Caloocan', 'Quezon', 'Manila', 'Pasig', 'Taguig', 'Makati',
    'Paranaque', 'Parañaque', 'Muntinlupa', 'Las Pinas', 'Las Piñas',
    'Marikina', 'Pasay', 'Valenzuela', 'Navotas', 'Malabon',
    'Mandaluyong', 'San Juan', 'Pateros'
]

BRANDS_ORDERED = [
    'PETRON', 'SHELL', 'CALTEX', 'PHOENIX', 'TOTAL',
    'FLYING V', 'UNIOIL', 'SEAOIL', 'PTT', 'INDEPENDENT'
]

BRAND_DISPLAY = {
    'PETRON': 'Petron', 'SHELL': 'Shell', 'CALTEX': 'Caltex',
    'PHOENIX': 'Phoenix', 'TOTAL': 'Total', 'FLYING V': 'Flying V',
    'UNIOIL': 'Unioil', 'SEAOIL': 'Seaoil', 'PTT': 'PTT',
    'INDEPENDENT': 'Independent'
}

PRODUCT_MAP = {
    'RON 100': 'RON 100', 'RON 97': 'RON 97', 'RON 95': 'RON 95',
    'RON 91': 'RON 91', 'DIESEL PLUS': 'Diesel Plus',
    'DIESEL': 'Diesel', 'KEROSENE': 'Kerosene'
}

In [3]:
def _derive_effectivity(s):
    if not s: return ''
    for pat in [
        r'([A-Za-z]+)\s+(\d+)\s*[-]\s*[A-Za-z]+\s+\d+,\s*(\d{4})',
        r'([A-Za-z]+)\s+(\d+)[-\s].*?(\d{4})',
        r'([A-Za-z]+)\s+(\d+),\s*(\d{4})',
    ]:
        m = re.search(pat, s)
        if m:
            try: return datetime.strptime(f"{m.group(1)[:3].capitalize()} {m.group(2)} {m.group(3)}", "%b %d %Y").strftime("%b %d, %Y")
            except Exception: pass
    return ''

def _normalize_city(raw):
    raw = raw.strip().replace('Paranaque', 'Parañaque').replace('Las Pinas', 'Las Piñas')
    raw = re.sub(r'\s*[Cc]ty\.?\s*$', ' City', raw)
    if not re.search(r'[Cc]ity\s*$', raw): raw += ' City'
    return raw.strip()

def _get_brand_positions(lines):
    for line in lines:
        if sum(1 for b in ['PETRON', 'SHELL', 'CALTEX'] if b in line.upper()) >= 2:
            positions = {}
            for brand in BRANDS_ORDERED:
                m = re.search(brand.replace(' ', r'\s+'), line, re.I)
                if m: positions[brand] = (m.start() + m.end()) / 2
            m_range = re.search(r'OVERALL\s*RANGE', line, re.I)
            return positions, (m_range.start() if m_range else 9999)
    return {}, 9999

def _extract_brand_prices(line, brand_centers, range_start):
    nums = [(m.start(), float(m.group())) for m in re.finditer(r'\d{2,3}\.\d{2}', line) if m.start() < range_start]
    if not nums: return {}
    brands, centers, bp = list(brand_centers.keys()), list(brand_centers.values()), {b: [] for b in brand_centers.keys()}
    for pos, val in nums:
        dists = [abs(pos - c) for c in centers]
        min_dist = min(dists)
        if min_dist < 18:
            nearest = brands[dists.index(min_dist)]
            if len(bp[nearest]) < 2: bp[nearest].append(val)
    return {b: v for b, v in bp.items() if v}

In [5]:
def check_pdf_integrity(data_folder='data'):
    """Scans a directory for PDF files and attempts to open them to verify integrity."""
    data_folder = Path(data_folder)
    pdf_files = list(data_folder.glob("*.pdf"))
    
    corrupt_files = []
    print(f"Scanning {len(pdf_files)} PDF files in '{data_folder}'...\n")

    for pdf_path in pdf_files:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                num_pages = len(pdf.pages)
                if num_pages == 0:
                    print(f"[WARNING] {pdf_path.name} is structurally valid but empty.")
        except Exception as e:
            print(f"[FAILED] {pdf_path.name} is corrupted. Error: {e}")
            corrupt_files.append(pdf_path)

    print("-" * 40)
    print(f"Valid Files: {len(pdf_files) - len(corrupt_files)}")
    print(f"Corrupt Files: {len(corrupt_files)}")
    
    return corrupt_files

# --- RUN THE CHECK ---
# BAD_FILES = check_pdf_integrity('path/to/your/pdf/folder')

In [13]:
integ_run = check_pdf_integrity(r'C:\Users\BrandonZahirDRosales\OneDrive - ASIAN INSTITUTE OF MANAGEMENT\Documents\DSA 2025\Data-Trio-Final-Project\00 Data Extraction\raw\doe_pdfs_v2')



Scanning 461 PDF files in 'C:\Users\BrandonZahirDRosales\OneDrive - ASIAN INSTITUTE OF MANAGEMENT\Documents\DSA 2025\Data-Trio-Final-Project\00 Data Extraction\raw\doe_pdfs_v2'...

[FAILED] petro_mm_2017_august_09.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_august_23.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_august_30.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_october_04.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_october_25.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_september_13.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_september_25.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_mm_2017_september_27.pdf is corrupted. Error: No /Root object! - Is this really a PDF?
[FAILED] petro_m

In [14]:
def delete_corrupted_files(corrupted_list):
    deleted_count = 0

    for file_path in corrupted_list:
        try:
            # Check if the file still exists just to be safe
            if file_path.exists():
                file_path.unlink() # This is the command that permanently deletes the file
                print(f"  [DELETED] {file_path.name}")
                deleted_count += 1
            else:
                print(f"  [SKIPPED] {file_path.name} (File already missing)")
                
        except Exception as e:
            print(f"  [ERROR] Could not delete {file_path.name}. Reason: {e}")

In [15]:
delete_corrupted_files(integ_run)

  [DELETED] petro_mm_2017_august_09.pdf
  [DELETED] petro_mm_2017_august_23.pdf
  [DELETED] petro_mm_2017_august_30.pdf
  [DELETED] petro_mm_2017_october_04.pdf
  [DELETED] petro_mm_2017_october_25.pdf
  [DELETED] petro_mm_2017_september_13.pdf
  [DELETED] petro_mm_2017_september_25.pdf
  [DELETED] petro_mm_2017_september_27.pdf
  [DELETED] petro_mm_2018_july_06.pdf
  [DELETED] petro_mm_2018_july_19.pdf


In [16]:
target_folder = "../00 Data Extraction/raw/doe_pdfs_v2" 
pdf_dir = Path(target_folder).resolve()

# Grab all PDF files in the directory
pdf_files = sorted(list(pdf_dir.glob("*.pdf")))


# Master container to hold data from ALL files
all_rows = []

# Loop through each file
for idx, pdf_path in enumerate(pdf_files, 1):
    print(f"[{idx}/{len(pdf_files)}] Extracting: {pdf_path.name}...")
    
    try:
        # A. Read the PDF
        pages_text = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                t = page.extract_text(layout=True)
                if t: pages_text.append(t)

        text = '\n'.join(pages_text)
        lines = text.split('\n')

        # B. Parse Dates 
        monitoring = effectivity = ''

        m_mon = re.search(r'Date of Monitoring[:\s]+([A-Za-z][\w\s,.\-]+?\d{4})', text)
        if m_mon: monitoring = re.sub(r'\s+', ' ', m_mon.group(1)).strip()
        
        if not monitoring:
            m_mon_alt = re.search(r'[Ff]or the week of\s+(.+?\d{4})', text[:800])
            if m_mon_alt: monitoring = re.sub(r'\s+', ' ', m_mon_alt.group(1)).strip().rstrip(')')

        effectivity_pattern = r'(?:Date of Effectivity|Effective Date|Effectivity Date|Effectivity|Effective)\s*[:\-]?\s*([A-Za-z][\w\s,]+?\d{4})'
        m_eff = re.search(effectivity_pattern, text, re.IGNORECASE)
        if m_eff:
            ds = re.sub(r'\s+', ' ', m_eff.group(1)).strip().replace('Januray', 'January')
            try: effectivity = datetime.strptime(ds, "%B %d, %Y").strftime("%b %d, %Y")
            except ValueError:
                try: effectivity = datetime.strptime(ds, "%b %d, %Y").strftime("%b %d, %Y")
                except ValueError: effectivity = ds
                
        if not effectivity:
            effectivity = _derive_effectivity(monitoring)

        # C. Calculate Column Alignment
        brand_centers, range_start = _get_brand_positions(lines)

        # D. Extract Row Data 
        city_pat = '|'.join(re.escape(c) for c in CITIES)
        
        # Container for THIS specific file
        file_rows = []

        # Regional Fallback for summary pages
        current_area = None
        if re.search(r'METRO\s*MANILA', text[:1000], re.IGNORECASE):
            current_area = "Metro Manila"

        for line in lines:
            cm = re.search(rf'^\s*({city_pat})\s*(City|Cty\.?)?', line, re.I)
            if cm:
                current_area = _normalize_city(cm.group(0).strip())

            pm = re.search(r'\b(RON\s*100|RON\s*97|RON\s*95|RON\s*91|DIESEL PLUS|DIESEL|KEROSENE)\b', line, re.I)
            
            if not pm or not current_area or not brand_centers:
                continue

            raw_prod = re.sub(r'\s+', ' ', pm.group(1).strip()).upper()
            product  = PRODUCT_MAP.get(raw_prod, raw_prod.title())

            prices_dict = _extract_brand_prices(line, brand_centers, range_start)

            for brand_key, prices in prices_dict.items():
                file_rows.append({
                    'Monitoring Dates': monitoring,
                    'Effectivity Date': effectivity,
                    'City':             current_area,
                    'Product':          product,
                    'Brand':            BRAND_DISPLAY[brand_key],
                    'Price Low (P/L)':  min(prices),
                    'Price High (P/L)': max(prices),
                    'Notes':            ''
                })
        
        # Add this file's rows to the master container
        all_rows.extend(file_rows)
        print(f"  -> Extracted {len(file_rows)} rows. (Effectivity: {effectivity})")

    except Exception as e:
        print(f"  -> ERROR processing {pdf_path.name}: {e}")

[1/451] Extracting: _petro_ncr_2021-apr-29.pdf...
  -> Extracted 358 rows. (Effectivity: Apr 27, 2021)
[2/451] Extracting: _petro_ncr_2021-june-10.pdf...
  -> Extracted 354 rows. (Effectivity: Jun 08, 2021)
[3/451] Extracting: ncr-price-monitoring-01062026.pdf...
  -> Extracted 299 rows. (Effectivity: Jan 06, 2026)
[4/451] Extracting: ncr-price-monitoring-01132026.pdf...
  -> Extracted 299 rows. (Effectivity: Jan 13, 2026)
[5/451] Extracting: ncr-price-monitoring-01202026.pdf...
  -> Extracted 403 rows. (Effectivity: Jan 20, 2026)
[6/451] Extracting: ncr-price-monitoring-01272026.pdf...
  -> Extracted 390 rows. (Effectivity: Jan 27, 2026)
[7/451] Extracting: ncr-price-monitoring-02032026.pdf...
  -> Extracted 403 rows. (Effectivity: Feb 03, 2026)
[8/451] Extracting: ncr-price-monitoring-02102026-1.pdf...
  -> Extracted 402 rows. (Effectivity: Feb 10, 2026)
[9/451] Extracting: ncr-price-monitoring-02172026.pdf...
  -> Extracted 401 rows. (Effectivity: Feb 17, 2026)
[10/451] Extracting: 

In [17]:
df_doe = pd.DataFrame(all_rows)

In [18]:
df_doe.head(25)

,Monitoring Dates,Effectivity Date,City,Product,Brand,Price Low (P/L),Price High (P/L),Notes
0,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Petron,52.40,52.40,
1,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Shell,56.85,56.85,
2,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Caltex,47.40,47.40,
3,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Phoenix,54.01,54.83,
4,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Total,56.80,56.80,
5,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Flying V,48.00,48.00,
6,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Unioil,52.17,52.17,
7,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Seaoil,53.60,53.60,
8,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,PTT,53.60,53.60,
9,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Independent,47.70,49.45,


In [19]:
df_doe.to_csv("../01 Data Preprocessing/processed_doe_data.csv", index=False, encoding='utf-8-sig')
df_doe.to_parquet("../01 Data Preprocessing/processed_doe_data.parquet", index=False)